In [ ]:
import time, joblib, gc, psutil
from math import sqrt
def mem():
    p = psutil.Process()
    return p.memory_info().rss/1024**2  # MB

def train_and_save(model, X_train, y_train, X_test, y_test, name):
    t0 = time.time()
    model.fit(X_train, y_train)
    t1 = time.time()
    # eval (regression or classification)
    try:
        preds = model.predict(X_test)
        from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
        if len(preds.shape)==1:  # regression
            metric = sqrt(mean_squared_error(y_test, preds))
            print(f"{name} RMSE: {metric:.6f}")
        else:
            print(f"{name} predictions shape: {preds.shape}")
    except Exception as e:
        print("Prediction/eval error:", e)
    joblib.dump(model, f"/content/{name}.joblib")
    print(f"Saved /content/{name}.joblib  (train secs: {t1-t0:.1f})")
    return t1 - t0

In [ ]:
# Optional: make a small stratified sample to test code quickly
sample_frac = 0.20   # 20% sample for quick testing; reduce if still heavy
Xs = X_train_s
ys_reg = y_reg_train
ys_clf = y_clf_train

# sample indices (stratify by label for classification stability)
import numpy as np
n = Xs.shape[0]
idx = np.random.RandomState(42).choice(n, size=int(n*sample_frac), replace=False)
X_train_s_small = Xs[idx]; y_reg_train_small = ys_reg[idx]; y_clf_train_small = y_clf_train[idx]

print("Sample shapes:", X_train_s_small.shape, y_reg_train_small.shape)
print("Mem before sample train (MB):", mem())

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso

# Linear Regression
t_lr = train_and_save(LinearRegression(), X_train_s, y_reg_train, X_test_s, y_reg_test, "linear_regression")

# Safely delete lr variable if it exists
if 'lr' in globals():
    del lr
gc.collect()
print("Mem after Linear (MB):", mem())

# Lasso Regression
t_lasso = train_and_save(Lasso(alpha=0.0005, max_iter=20000), X_train_s, y_reg_train, X_test_s, y_reg_test, "lasso_reg")

gc.collect()
print("Mem after Lasso (MB):", mem())

In [ ]:
from sklearn.linear_model import LogisticRegression
logr = LogisticRegression(max_iter=2000, solver="saga")
t_logr = train_and_save(logr, X_train_s, y_clf_train, X_test_s, y_clf_test, "logistic")
# explicit delete and collect
del logr
gc.collect()
print("Mem after Logistic (MB):", mem())

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import joblib

# Load the saved logistic model
logr = joblib.load("/content/logistic.joblib")

y_pred = logr.predict(X_test_s)
acc = accuracy_score(y_clf_test, y_pred)
f1  = f1_score(y_clf_test, y_pred)

print(f"✅ Logistic Regression — Accuracy: {acc:.4f}, F1-score: {f1:.4f}")

In [ ]:
import joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize

# Load your saved RF Regressor model
rf_reg = joblib.load("/content/rf_regressor.joblib")
print("✅ RF Regressor model loaded.")

# We'll reuse your train/test DataFrames (train_df, test_df, etc.)
# If not in memory, re-read from the combined parquet subset:
# test_ddf = dd.read_parquet("/content/drive/MyDrive/stocks_processed_parquet_combined_fixed/")
# pandas_df = test_ddf.compute()
# (ensure you have train_df and test_df ready from earlier cells)


# Predict future returns on the training set to estimate expected μ for each ticker
train_df2 = train_df.copy()
train_df2["PredReturn_RF"] = rf_reg.predict(X_train_s)

# Mean predicted return per ticker
mu_pred = train_df2.groupby("Ticker")["PredReturn_RF"].mean()
# Historical mean returns per ticker
mu_hist = train_df2.groupby("Ticker")["DailyReturn"].mean()

tickers = mu_hist.index.intersection(mu_pred.index)
mu_pred_vec = mu_pred.loc[tickers].values
mu_hist_vec = mu_hist.loc[tickers].values
print("✅ Expected returns computed for", len(tickers), "tickers.")


# Fix: aggregate duplicates before pivoting
pivot_train = (
    train_df.groupby(["Date", "Ticker"])["DailyReturn"]
    .mean()
    .reset_index()
    .pivot(index="Date", columns="Ticker", values="DailyReturn")
)

# Align columns with your selected tickers
pivot_train = pivot_train.reindex(columns=tickers)

# Compute covariance matrix
cov = pivot_train.cov().values
print("✅ Covariance matrix built with shape:", cov.shape)


def portfolio_stats(weights, mean_returns, cov_matrix, rf=0.0):
    w = np.array(weights)
    ret = np.dot(w, mean_returns)
    vol = np.sqrt(np.dot(w, cov_matrix.dot(w)))
    sharpe = (ret - rf) / vol if vol > 0 else 0
    return ret, vol, sharpe

def negative_sharpe(weights, mean_returns, cov_matrix):
    return -portfolio_stats(weights, mean_returns, cov_matrix)[2]

def get_markowitz_weights(mean_returns, cov_matrix, long_only=True):
    n = len(mean_returns)
    x0 = np.ones(n) / n
    bounds = tuple((0, 1) if long_only else (-1, 1) for _ in range(n))
    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1},)
    res = minimize(negative_sharpe, x0, args=(mean_returns, cov_matrix),
                   method='SLSQP', bounds=bounds, constraints=cons)
    if not res.success:
        print("⚠ Optimization Warning:", res.message)
    return res.x


# Optimize portfolios (Predicted vs Historical)
w_pred = get_markowitz_weights(mu_pred_vec, cov, long_only=True)
w_hist = get_markowitz_weights(mu_hist_vec, cov, long_only=True)

print("✅ Markowitz optimization complete.")
print("Top 10 weights (Predicted):")
for t, w in zip(tickers[:10], w_pred[:10]):
    print(f"{t}: {w:.4f}")

In [ ]:
# ============================================
# ✅ Backtest Portfolios (with duplicate-safe pivot)
# ============================================

# Fix: aggregate duplicates before pivoting (mean of DailyReturn per Date/Ticker)
pivot_test = (
    test_df.groupby(["Date", "Ticker"])["DailyReturn"]
    .mean()
    .reset_index()
    .pivot(index="Date", columns="Ticker", values="DailyReturn")
)
pivot_test = pivot_test.reindex(columns=tickers)

# --- Define backtest function ---
def backtest(weights, pivot_returns):
    R = pivot_returns.fillna(0.0).values
    daily = R.dot(weights)

    # Compute portfolio statistics
    cum_ret = np.prod(1 + daily) - 1
    ann_ret = (1 + cum_ret) ** (252 / len(daily)) - 1
    vol = np.std(daily) * np.sqrt(252)
    sharpe = (np.mean(daily) * 252) / vol if vol > 0 else 0

    return {
        "Cumulative Return": cum_ret,
        "Annual Return": ann_ret,
        "Annual Volatility": vol,
        "Sharpe": sharpe
    }

# --- Run backtests ---
res_pred = backtest(w_pred, pivot_test)
res_hist = backtest(w_hist, pivot_test)

# --- Display results ---
print("✅ Backtest Results:")
print("Predicted Portfolio:", res_pred)
print("Historical Portfolio:", res_hist)